# Colab GPU Embedding + Rerank Server

Runs a strong 1024-d Vietnamese embedding model (and an optional Vietnamese reranker) on a **Colab GPU**, exposed through a **cloudflared** tunnel. The local `model-service` proxies `/embed` (and optionally `/rerank`) here, so every caller (backend pipeline, `reembed.py`, `enrich`, `load_corpus`) keeps hitting `model-service:8001` unchanged while the heavy compute runs on GPU.

**Endpoints (match model-service's contract exactly):**
- `POST /embed`  `{"texts":[...]}` → `{"embeddings":[[...]], "model_id":null}`
- `POST /rerank` `{"query":..., "documents":[...], "top_n":N}` → `{"results":[{"index","relevance_score"}]}`
- `GET /v1/models`, `GET /health` (probe)

All POST routes require `Authorization: Bearer <API_KEY>`.

**Setup:** Runtime → Change runtime type → **GPU** (T4 is enough), then run the cells top to bottom. The last cell prints `TUNNEL_URL` + `API_KEY` — give those to the local side and recreate `model-service` with `REMOTE_EMBED_URL` / `REMOTE_API_KEY` set.

In [ ]:
!pip -q install sentence-transformers fastapi "uvicorn[standard]"

In [ ]:
# ── Config + load models on GPU ───────────────────────────────────────────
import os, secrets, torch
from sentence_transformers import SentenceTransformer, CrossEncoder

PORT         = 8000
EMBED_MODEL  = os.environ.get("EMBED_MODEL",  "AITeamVN/Vietnamese_Embedding")   # bge-m3 based, 1024-d
RERANK_MODEL = os.environ.get("RERANK_MODEL", "AITeamVN/Vietnamese_Reranker")    # set "" to disable
API_KEY      = os.environ.get("EMBED_API_KEY", "kb-emb-" + secrets.token_hex(8))
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

print(f"loading embed={EMBED_MODEL} rerank={RERANK_MODEL or '(none)'} on {DEVICE} …", flush=True)
_embed  = SentenceTransformer(EMBED_MODEL, device=DEVICE)
_rerank = CrossEncoder(RERANK_MODEL, device=DEVICE) if RERANK_MODEL else None
_dim    = _embed.get_sentence_embedding_dimension()
print(f"ready: embed dim={_dim}", flush=True)

In [ ]:
# ── FastAPI app (contract matches local model-service) ─────────────────────
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel

app = FastAPI(title="colab-embed-rerank")

def _auth(authorization):
    if API_KEY and authorization != f"Bearer {API_KEY}":
        raise HTTPException(401, "bad api key")

class EmbedReq(BaseModel):
    texts: list[str]
    model_id: str | None = None

@app.post("/embed")
def embed(req: EmbedReq, authorization: str | None = Header(default=None)):
    _auth(authorization)
    if not req.texts:
        return {"embeddings": [], "model_id": None}
    # normalize_embeddings=True to match local model-service (cosine-ready vectors)
    vecs = _embed.encode(req.texts, normalize_embeddings=True, batch_size=64,
                         convert_to_numpy=True, show_progress_bar=False)
    return {"embeddings": vecs.tolist(), "model_id": None}

class RerankReq(BaseModel):
    query: str
    documents: list[str]
    top_n: int = 5

@app.post("/rerank")
def rerank(req: RerankReq, authorization: str | None = Header(default=None)):
    _auth(authorization)
    if _rerank is None:
        raise HTTPException(503, "reranker disabled on this server")
    if not req.documents:
        return {"results": []}
    scores = _rerank.predict([(req.query, d) for d in req.documents]).tolist()
    ranked = sorted(({"index": i, "relevance_score": float(s)} for i, s in enumerate(scores)),
                    key=lambda x: x["relevance_score"], reverse=True)[: min(req.top_n, len(scores))]
    return {"results": ranked}

@app.get("/v1/models")
def models():
    return {"data": [{"id": EMBED_MODEL, "dim": _dim, "rerank": RERANK_MODEL or None}]}

@app.get("/health")
def health():
    return {"status": "ok", "dim": _dim}

In [ ]:
# ── Serve + cloudflared tunnel (prints TUNNEL_URL + API_KEY) ────────────────
import re, time, threading, subprocess, urllib.request, uvicorn

threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning"),
                 daemon=True).start()
time.sleep(3)

if not os.path.exists("./cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "cloudflared")
    os.chmod("cloudflared", 0o755)

cf_log = open("cloudflared.log", "w")
subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}",
                  "--no-autoupdate"], stdout=cf_log, stderr=subprocess.STDOUT)

TUNNEL_URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
    if m:
        TUNNEL_URL = m.group(0); break
    time.sleep(1)
assert TUNNEL_URL, "tunnel URL not found — check cloudflared.log"

print("=" * 70)
print("  TUNNEL_URL :", TUNNEL_URL)
print("  API_KEY    :", API_KEY)
print("  EMBED_MODEL:", EMBED_MODEL, f"(dim {_dim})")
print("  RERANK     :", RERANK_MODEL or "(disabled)")
print("=" * 70)
print(f"\n  test:  !curl {TUNNEL_URL}/health")

In [ ]:
# ── Keep the runtime alive (Ctrl-C / stop to shut the tunnel) ───────────────
import time
while True:
    time.sleep(3600)